In [1]:
!pip -q install gradio easyocr
!ffmpeg -version | head -1

ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers


In [2]:

import os, shutil, sys, glob

# 1. Copy the app folder
src = glob.glob(
    '/kaggle/input/datasets/sanawerb/final-tikharm-app/tikharm_appp'
)

assert src, "Could not find the tikharm_appp folder. Check the attached dataset path."

APP_DIR = '/kaggle/working/tikharm_appp'
shutil.copytree(src[0], APP_DIR, dirs_exist_ok=True)

# 2. Find the saved model checkpoint
checkpoint_src = (
    '/kaggle/input/datasets/sanawerb/artifacts/'
    'kaggle/working/artifacts/final_head.pt'
)

assert os.path.isfile(checkpoint_src), (
    f"Checkpoint not found at: {checkpoint_src}"
)

# 3. Put it where the app expects it
os.makedirs(f'{APP_DIR}/artifacts', exist_ok=True)

shutil.copy2(
    checkpoint_src,
    f'{APP_DIR}/artifacts/final_head.pt'
)

# 4. Set Python import path and working directory
sys.path.insert(0, APP_DIR)
os.chdir(APP_DIR)

print("App directory:", APP_DIR)
print("Checkpoint exists:",
      os.path.isfile(f'{APP_DIR}/artifacts/final_head.pt'))
print("Setup complete!")


App directory: /kaggle/working/tikharm_appp
Checkpoint exists: True
Setup complete!


In [3]:

from pathlib import Path

model_file = Path("/kaggle/working/tikharm_appp/model.py")

code = model_file.read_text()

old = 'v.float().numpy()'
new = 'v.float().cpu().numpy()'

assert old in code, "Expected line not found. Inspect model.py before changing it."

model_file.write_text(code.replace(old, new))

print("Patched model.py:", new in model_file.read_text())


Patched model.py: True


In [4]:

import torch
import app
import model

print("Model module loaded from:", model.__file__)

# Check the actual patched code
with open(model.__file__, "r") as f:
    for i, line in enumerate(f, 1):
        if 50 <= i <= 60:
            print(f"{i}: {line.rstrip()}")

# Inspect scaler values directly from the checkpoint
ck = torch.load(
    "/kaggle/working/tikharm_appp/artifacts/final_head.pt",
    map_location="cpu",
    weights_only=False
)

print("\nScaler details:")
for k, v in ck["scalers"].items():
    print(
        k,
        "| type:", type(v),
        "| device:", getattr(v, "device", "not a tensor"),
        "| value:", v
    )


Model module loaded from: /kaggle/working/tikharm_appp/model.py
50: 
51: def load_head(path, device="cpu"):
52:     ck = torch.load(path, map_location=device, weights_only=True)
53:     nets = []
54:     for st in ck["states"]:
55:         n = Net3(ck["mode"], ck["use"], ck["cfg"]["d"], ck["cfg"]["drop"]).to(device)
56:         n.load_state_dict(st); n.eval(); nets.append(n)
57:     sc = {k: v.float().cpu().numpy() for k, v in ck["scalers"].items()}
58:     return nets, sc, ck
59: 
60: @torch.no_grad()

Scaler details:
mu_v | type: <class 'torch.Tensor'> | device: cpu | value: tensor([-4.4916e-02, -8.3863e-02, -3.3751e-02,  9.1277e-03, -1.3773e-02,
        -7.1173e-03, -5.3836e-02,  5.4055e-02,  9.6079e-04, -3.6586e-02,
        -7.3679e-02, -8.1523e-02,  1.1506e-01, -3.1096e-02, -3.2496e-02,
        -2.4836e-02, -1.2308e-03, -8.7268e-02, -2.5620e-02,  9.0086e-03,
        -3.7400e-02, -2.9114e-02,  4.3069e-02, -4.7559e-02, -2.5737e-02,
        -5.9158e-02,  1.1712e-01, -8.0211e-02, -7.0

In [5]:

import torch
import model

path = "/kaggle/working/tikharm_appp/artifacts/final_head.pt"

# Test the actual loader directly
try:
    nets, scalers, ck = model.load_head(path, device="cpu")
    print("CPU loader: SUCCESS")
    print("Scaler devices:", {
        k: str(v.device) for k, v in ck["scalers"].items()
    })
    print("Scaler types:", {
        k: type(v).__name__ for k, v in scalers.items()
    })
except Exception as e:
    print("CPU loader FAILED:", repr(e))

# Now test using the current runtime's GPU, if available
if torch.cuda.is_available():
    try:
        nets, scalers, ck = model.load_head(path, device="cuda")
        print("GPU loader: SUCCESS")
    except Exception as e:
        print("GPU loader FAILED:", repr(e))


CPU loader: SUCCESS
Scaler devices: {'mu_v': 'cpu', 'sd_v': 'cpu', 'mu_a': 'cpu', 'sd_a': 'cpu', 'mu_t': 'cpu', 'sd_t': 'cpu'}
Scaler types: {'mu_v': 'ndarray', 'sd_v': 'ndarray', 'mu_a': 'ndarray', 'sd_a': 'ndarray', 'mu_t': 'ndarray', 'sd_t': 'ndarray'}
GPU loader: SUCCESS


In [6]:
import os
import shutil

src = "/kaggle/input/datasets/sanawerb/final-tikharm-app/tikharm_appp/prototype_policy (1).md"
dst = "/kaggle/working/tikharm_appp/prototype_policy.md"

assert os.path.isfile(src), f"Policy file not found: {src}"

shutil.copy2(src, dst)

print("Policy file copied:", os.path.isfile(dst))
print("App can now find:", dst)

Policy file copied: True
App can now find: /kaggle/working/tikharm_appp/prototype_policy.md


In [7]:
import os
import sys

APP_DIR = "/kaggle/working/tikharm_appp"
os.chdir(APP_DIR)

if APP_DIR not in sys.path:
    sys.path.insert(0, APP_DIR)

import app

moderator = app.build_moderator()
demo = app.create_app(moderator)

demo.queue().launch(share=True)



preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.87k [00:00<?, ?B/s]

encoder status: {'vit': 'ok', 'ast': 'ok', 'text_encoder': 'ok', 'whisper': 'ok', 'ocr': 'ok'}


/kaggle/working/tikharm_appp/app.py:98: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(title="TriSensus - Version 2", theme=custom_theme, css=css) as demo:


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://99d69e47da074a4bdc.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
